# Controlled private replay: explicit stages only
Child `qwen3-controlled-specialization-001`. Review/push code and pin the full commit.
Default: plan only, zero model calls. No parent recollection, training sweep or GPQA.
The specification requires a later-development-use review before planning. Set the
review parameter only after checking that the original32 development groups have
not been used for optimization or outcome-based choices since the parent stopped.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, re
from google.colab import drive
GIT_REF = "PASTE_REVIEWED_FULL_COMMIT_SHA"
STAGE = "plan"  # plan/acquire/build-bank/replay/score-assign/train/evaluate/report/export/restore
EXECUTE = False
PLAN_HASH = ""
ARM = "frozen"
RESUME = False
STOP_AFTER = None
DEVELOPMENT_USE_REVIEWED = False  # explicit prerequisite, not a model result
DEVELOPMENT_REVIEW_NOTE = ""  # who checked which records / any external uses
LATER_DEVELOPMENT_USES = []  # any optimization/outcome-selection use stops planning
SCRATCH = Path("/content/pact-scratch")
CHECKOUT = SCRATCH / "checkout"
DRIVE = Path("/content/drive/MyDrive/PACT")
RUN = SCRATCH / "controlled-specialization/qwen3-controlled-specialization-001"
PERSISTENT = DRIVE / "controlled-specialization/qwen3-controlled-specialization-001"
PARENT_NAME = "qwen3-specialization-contrast-001-handoff-1790416052208683365.zip"
PARENT_DRIVE = DRIVE / "specialization/qwen3-specialization-contrast-001/bundles" / PARENT_NAME
PARENT = SCRATCH / "sources/controlled-specialization-parent.zip"
PARENT_SHA = "555d9bd2e724c59ca4a91db156d450f814d0149ed6413922e965c3a85fb4e69f"
INITIAL = SCRATCH / "actor-preparation/qwen3-preparation-120-001-inference"
PREP_SNAPSHOT = DRIVE / "actor-preparation/qwen3-preparation-120-001/training/snapshots/1789930640656395583-479df47c14d9"
REVIEW_FILE = SCRATCH / "controlled-training-review.json"
assert re.fullmatch(r"[0-9a-fA-F]{40}", GIT_REF)
drive.mount("/content/drive")
SCRATCH.mkdir(parents=True, exist_ok=True)
def git(*args):
    return subprocess.check_output(["git", *args], text=True).strip()
if not CHECKOUT.exists():
    subprocess.run(["git", "clone", "https://github.com/Soqoro/Pact.git", str(CHECKOUT)], check=True)
assert not git("-C", str(CHECKOUT), "status", "--porcelain"), "Preserve local changes."
assert git("-C", str(CHECKOUT), "remote", "get-url", "origin") == "https://github.com/Soqoro/Pact.git"
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", GIT_REF], check=True)
assert git("-C", str(CHECKOUT), "rev-parse", "HEAD").lower() == GIT_REF.lower()
os.chdir(CHECKOUT)
sys.path.insert(0, str(CHECKOUT / "src"))
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
from pact.colab import install_dependencies
install_dependencies(CHECKOUT)
from pact.storage import storage_operation
from pact.util import read_json, write_json, file_hash, digest
if not PARENT.exists():
    assert PARENT_DRIVE.is_file(), f"Required parent ZIP missing: {PARENT_DRIVE}"
    PARENT.parent.mkdir(parents=True, exist_ok=True)
    print(storage_operation("bundle-restore", PARENT_DRIVE, PARENT,
                            sha256=PARENT_SHA, timeout_seconds=600))
assert file_hash(PARENT) == PARENT_SHA

def study(stage, *options):
    command = [sys.executable, "-u", "-m", "pact.training.controlled_specialization",
               stage, "--run-dir", str(RUN), "--persistent", str(PERSISTENT),
               *map(str, options)]
    log = SCRATCH / f"controlled-{stage}-{__import__('time').time_ns()}.log"
    print("Starting:", stage, "Log:", log, flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    with log.open("w") as stream:
        for line in process.stdout:
            print(line, end="", flush=True)
            stream.write(line)
            stream.flush()
    if process.wait():
        raise RuntimeError(f"{stage} stopped. Preserve scratch/attempts and return {log}.")


In [ ]:
if STAGE == "restore":
    assert not RUN.exists(), "Restore requires fresh scratch; never erase partial work."
    snapshots = sorted(p for p in (PERSISTENT / "snapshots").glob("*") if p.is_dir())
    assert snapshots, "No full child snapshot. Review ZIPs are not resumable weights."
    study("restore", "--snapshot", snapshots[-1])
elif STAGE == "plan":
    assert DEVELOPMENT_USE_REVIEWED and DEVELOPMENT_REVIEW_NOTE, "Complete the later-development-use review required by specification section4."
    exposure = SCRATCH / "controlled-development-exposure.json"
    write_json(exposure, {"reviewed": DEVELOPMENT_USE_REVIEWED,
                         "provenance": DEVELOPMENT_REVIEW_NOTE,
                         "development_used_for_optimization_or_outcome_selection": LATER_DEVELOPMENT_USES})
    study("plan", "--parent-bundle", PARENT, "--exposure-review", exposure)
    PLAN_HASH = digest(read_json(RUN / "plan.json"))
    print("Frozen PLAN_HASH:", PLAN_HASH)
else:
    assert re.fullmatch(r"[0-9a-f]{64}", PLAN_HASH), "Use the printed child plan hash."
    assert digest(read_json(RUN / "plan.json")) == PLAN_HASH
    options = ["--plan-hash", PLAN_HASH]
    if STAGE in ("acquire", "replay", "score-assign", "train", "evaluate"):
        assert EXECUTE, "Explicitly select this one GPU stage."
        from pact.training.preparation_restore import restore_preparation_references, verify_inference_export
        if not INITIAL.exists():
            print(restore_preparation_references(PREP_SNAPSHOT, INITIAL, timeout_seconds=1800))
        verify_inference_export(INITIAL)
        options += ["--execute", "--initialization-root", INITIAL, "--cache-dir", SCRATCH / "cache"]
        if STOP_AFTER is not None:
            options += ["--stop-after", str(STOP_AFTER)]
        if STAGE in ("train", "evaluate"):
            options += ["--arm", ARM]
            if REVIEW_FILE.exists(): options += ["--review", REVIEW_FILE]
        if RESUME: options.append("--resume")
    study(STAGE, *options)


Stop after each requested stage. Order: plan → acquire → build-bank/support review →
replay (primary + fixed controls) → score-assign → **export and user review**.
Only after explicit review: train one named arm at a time, then evaluate frozen
and each final trained arm, report, export. Never paste an all-arm loop into Run All.
Support/practical gate failure is a valid stop: export, do not add draws/tasks.
Training review format and individual commands: `docs/controlled_private_colab.md`.
After runtime loss restore the LATEST full child snapshot; never roll back attempts.
